In [30]:
import seaborn as sns
titanic = sns.load_dataset("titanic")  

In [31]:
print(titanic["deck"].value_counts())
print(titanic["deck"].isna().sum())

deck
C    59
B    47
D    33
E    32
A    15
F    13
G     4
Name: count, dtype: int64
688


In [32]:
titanic_propre = titanic.drop(columns=["deck"])
print(titanic_propre.shape)
print(titanic_propre["age"].isna().sum()) 
mediane_age = titanic_propre.groupby(["pclass", "sex"])["age"].transform("median")
titanic_propre["age"] = titanic_propre["age"].fillna(mediane_age) 
print(titanic_propre["age"].isna().sum()) 
titanic_propre["embark_town"] = titanic_propre["embark_town"].fillna("Inconnue") 
print(titanic_propre["embark_town"].isna().sum())
          

 

(891, 14)
177
0
0


## Journal de nettoyage

| Quoi | Pourquoi | Lignes |
|---|---|---|
| `deck` supprimée (colonne entière) | plus de la moitié de valeurs manquantes : 688 sur 891 (77 %) | 891 |
| `age` rempli par la médiane de sa classe et de son sexe | 177 manquants (20 %) ; l'âge est utile, on garde la colonne | 177 |
| `embark_town` : manquants remplacés par "Inconnue" | seulement 2 manquants ; je ne veux pas supprimer une ligne entière pour 2 cases vides | 2 |
| Doublons gardés | 118 lignes identiques, mais aucune preuve que ce sont des copies | 0 |
| `alive`, `class` et `embarked` supprimées | colonnes redondantes : chacune redit une autre colonne ; je garde `survived`, `pclass` et `embark_town` car elles sont plus simples à utiliser et je les ai déjà utilisées ou nettoyées | 891 |
| `sex`, `who` et `embark_town` converties en type `category` | colonnes de texte avec très peu de valeurs différentes (2, 3 et 4)   | 891 |
| Prix très élevés gardés (116 billets au-dessus de 65,6) | `fare` au-dessus de la limite de l'IQR (Q3 + 1,5 × écart), mais en 1re classe de tels prix sont normaux : valeurs inhabituelles, pas fausses | 0 |

In [34]:
print(titanic_propre.duplicated().sum())
lignes_doublons = titanic_propre[titanic_propre.duplicated(keep=False)]
print(len(lignes_doublons)) 
print(lignes_doublons.sort_values(["pclass", "sex", "age", "fare"]).head(10))

118
175
     survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
369         1       1  female  24.0      0      0  69.3000        C  First   
641         1       1  female  24.0      0      0  69.3000        C  First   
383         1       1  female  35.0      1      0  52.0000        S  First   
669         1       1  female  35.0      1      0  52.0000        S  First   
263         0       1    male  40.0      0      0   0.0000        S  First   
633         0       1    male  40.0      0      0   0.0000        S  First   
815         0       1    male  40.0      0      0   0.0000        S  First   
30          0       1    male  40.0      0      0  27.7208        C  First   
64          0       1    male  40.0      0      0  27.7208        C  First   
295         0       1    male  40.0      0      0  27.7208        C  First   

       who  adult_male  embark_town alive  alone  
369  woman       False    Cherbourg   yes   True  
641  woman       False    Cherb

In [35]:
titanic_propre = titanic_propre.drop(columns=["alive","class","embarked"])
print(titanic_propre.shape)

(891, 11)


In [ ]:

titanic_propre["sex"] = titanic_propre["sex"].astype("category") 
titanic_propre["who"] = titanic_propre["who"].astype("category") 
titanic_propre["embark_town"] = titanic_propre["embark_town"].astype("category") 
print(titanic_propre.dtypes)

survived          int64
pclass            int64
sex            category
age             float64
sibsp             int64
parch             int64
fare            float64
who            category
adult_male         bool
embark_town    category
alone              bool
dtype: object


In [50]:
q1 = titanic_propre["fare"].quantile(0.25)
q3 = titanic_propre["fare"].quantile(0.75)
print(q1)
print(q3)
ecart = q3 - q1
borne_haute = q3 + 1.5 * ecart
print(ecart)
print(borne_haute)
tres_chers = titanic_propre[titanic_propre["fare"]> borne_haute]
print(len(tres_chers))
print(tres_chers["pclass"].value_counts())
print(tres_chers.sort_values("fare" , ascending=False)[["fare","pclass","embark_town"]].head(5))



7.9104
31.0
23.0896
65.6344
116
pclass
1    104
3      7
2      5
Name: count, dtype: int64
         fare  pclass  embark_town
258  512.3292       1    Cherbourg
737  512.3292       1    Cherbourg
679  512.3292       1    Cherbourg
27   263.0000       1  Southampton
341  263.0000       1  Southampton


In [51]:
titanic_propre.to_csv("data/titanic_propre.csv", index=False) 
print(titanic_propre.to_csv)

<bound method NDFrame.to_csv of      survived  pclass     sex   age  sibsp  parch     fare    who  adult_male  \
0           0       3    male  22.0      1      0   7.2500    man        True   
1           1       1  female  38.0      1      0  71.2833  woman       False   
2           1       3  female  26.0      0      0   7.9250  woman       False   
3           1       1  female  35.0      1      0  53.1000  woman       False   
4           0       3    male  35.0      0      0   8.0500    man        True   
..        ...     ...     ...   ...    ...    ...      ...    ...         ...   
886         0       2    male  27.0      0      0  13.0000    man        True   
887         1       1  female  19.0      0      0  30.0000  woman       False   
888         0       3  female  21.5      1      2  23.4500  woman       False   
889         1       1    male  26.0      0      0  30.0000    man        True   
890         0       3    male  32.0      0      0   7.7500    man        True